<span style="font-size:10pt">RDVS Numériques" du département DuMAS de l’I2M / Mars 5, 2026<br> 
v1.0  - CC BY-SA 4.0 Jean-Luc CHARLES (Jean-Luc.charles@mailo.com)</span>

<div class="alert alert-block alert-danger">
<span style="color:brown;font-family:arial;font-size:normal">
     It is important to define a <span style="font-weight:bold;">Python Virtual Environment</span> (PVE) for each Python project: a PVE makes it possible to control for each project the versions of the Python interpreter and “sensitive” modules (like tensorflow for example).<br>
$\leadsto$ This notebook is run with the command `uv run jupyter lab` to ensure it uses the PVE of the projet.

<span style="font-family:arial;font-size:1cm;">
    Machine learning with tensorflow2/keras Python modules
</span>

# Training a Convolutional Neural Network to classify material micrographs

## 0 - Preliminaries

### Import the required Python modules

In [ ]:
# suppress tensorflow verbose warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
# Deep Learning modules:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# General modules:
import numpy as np
import matplotlib.pyplot as plt
import sys
import random
import cv2
from time import time
from pathlib import Path
from cpuinfo import get_cpu_info

# Custom modules:
from utils.tools import scan_dir, plot_images, plot_loss_accuracy, elapsed_time_since, show_conf_matrix, plot_proportion_bar
from utils.tools import split_stratified_into_train_val_test

In [ ]:
print(f"Python    : {sys.version.split()[0]}")
print(f"tensorflow: {tf.__version__} with keras {keras.__version__}")
print(f"numpy     : {np.__version__}")
print(f"OpenCV    : {cv2.__version__}")

In [ ]:
# allows to visualize the graphs directly in the cell of the N.B.
%matplotlib inline

# SEED will be used to fix the _seed_ of the random generators to have continuations
# of repeatable random numbers
SEED = 11

### Create the `models` directory

In [ ]:
if Path.cwd().name != 'Notebooks':
    print("You should save & close this notebook and type 'uv run jupyter lab'")
    print("from the repository root directory ('Micrograph_Classification')")
else:
    model_path = Path("./models")
    model_path.mkdir(exist_ok=True)
    print(f"Directory `{model_path.name} successfuly created.")

## 1 - The image files in the directory `../Data`

The directory `Data` in the root directory contains the image files:

In [ ]:
os.listdir('..')

Lest's define `data_dir` : the relative path to the data directory:

In [ ]:
data_dir = Path('../Data')

Now we build the ordered list of the subdirs in `data_dir`:

In [ ]:
data_subdirs = sorted(list(data_dir.iterdir()))
data_subdirs

### Check: display some images

Let's display the first JPG files in the three image directories: we read the JPG files with the function `imread` of the `openCV` module, wich returns a `ndarray` of the image pixels:

In [ ]:
fig, axes = plt.subplots(1, 3)
fig.set_size_inches((12,3))

for subdir, axis in zip(data_subdirs, axes):
    files = sorted(subdir.glob('**/*.jpg'))   # '**/*.jpg' means: all the *.jpg files in 'subdir'
    imBGR = cv2.imread(files[0])              # load the image file
    imRGB = cv2.cvtColor(imBGR, cv2.COLOR_BGR2RGB)
    # display images:
    axis.imshow(imRGB)
    axis.set_title(f'{subdir}\n ndarray shape: {imRGB.shape}', fontsize=10) 
    axis.axis('off')
    axis.axis('equal')

### Check: size of the images

In [ ]:
shapes = []
for subdir in data_subdirs:
    files = sorted(subdir.glob('**/*.jpg'))              # '**/*.jpg' means: all the *.jpg files in 'subdir'
    print(f'collecting image size in subdir "{subdir.name}"')
    for file in files:
        img = cv2.imread(file)
        h, w, _ = img.shape
        shapes.append((w, h))
print(f'Found image sizes: {set(shapes)}')

All the images have not the same size => we will have to resize the images to a common size.

## 2 - Preprocessing of the images

__Size__:<br>
- all the images must have the same size
- the size of the images should not be too large if we want acceptable computation times...

$\leadsto$ the micrographs available are high resolution images ( 2048$\times$1536 and 2880$\times$2160 ): we will lower the image size with the fucntion `resize` of the `openCV` module.

__RGB__  or __gray tones__:<br>
If the color information of the images is not relevant for their classification, we can gain calculation time and minimize the memory footprint by converting the images into gray tones.

$\leadsto$ As the color is not the main classification information, lets's convert the RGB images to gray-tone images with the `cvtColor` function of OpenCV.

### Example:

Let's resize and convert in gray the first images of the three image sub-directories:

In [ ]:
fig, axes = plt.subplots(1, 3)
fig.set_size_inches((12,3))

w_target, h_target = 400, 300
for subdir, axis in zip(data_subdirs, axes):
    files = sorted(subdir.glob('**/*.jpg'))           # '**/*.jpg' means: all the *.jpg files in 'subdir'
    img = cv2.imread(files[0])                        # load the first image
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)       # convert color image into grayscale image
    img = cv2.resize(img, dsize=(w_target, h_target)) # new size given as: (width, height)
    axis.imshow(img, cmap='gray')
    axis.set_title(f'{subdir}\n ndarray shape: {img.shape}', fontsize=10) 
    axis.axis('off')
    axis.axis('equal')

### Building the data set

Now:<br>
- we load into a single ndarray all the images resized and converted in gray tone<br>
- we build the `labels` ndarray.

In [ ]:
data_dir     = Path('../Data')  # the data directory
data_subdirs = sorted(list(data_dir.iterdir()))

# Initiallize empty lists:
images, labels = [], []
label_rank, label_text = [], []

for rank, subdir in enumerate(data_subdirs):
    label_rank.append(rank)
    label_text.append(subdir.name)
    print(f'Building dataset for material: label:{rank}, subdir: {subdir.name}')
    files = list(subdir.glob("*.jpg"))                    # '**/*.jpg' means: all the *.jpg files in 'subdir'
    files.sort()
    for f in files:
        print(f'\r\treading file {f.name}', end='')
        img = cv2.imread(f)                               # load the first image
        img = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)        # convert color image into grayscale image
        img = cv2.resize(img, dsize=(w_target, h_target)) # new size given as: (width, height)
        
        images.append(img)
        labels.append(rank)
    print(f'\r\tdone{40*" "}')
    
# convert lists as ndarrays:    
images = np.array(images)
labels = np.array(labels)

In [ ]:
label_text, list(label_rank)

How many image in total:

In [ ]:
len(images)

#### Check the size of the first and last images:

In [ ]:
images[0].shape, images[-1].shape

Summary:

In [ ]:
print(f'{images.shape=}, {images.dtype=}')
print(f'{labels.shape=}, {labels.dtype=}')
print(f"total size of the {len(images)} images ndarray in memory: {images.size/1024/1024:.1f} MiB")

## 3 - Prepare the _train_, _valid_ and _test_ data sets

### Split the full datset into train, valid & test datasets

Thanks to the `train_test_split` function of the `sklearn` module we can split the `images` and `labels` ndarrays into sub-datasets.<br>
Images and labels are randomly selected but respecting the proportion of each of the 3 classes in the original dataset (this is the interest of the `stratify` argument of the `train_test_split` function).

Following the state of the art in Deep Learning, we will split the full dataset into 3 sub-datasets:
- a `train` dataset (for the training)
- a `valid` dataset (for the validation during the model training)
- a 'test' dataset to evaluate the model score after training.

$\leadsto$ See the fucntion `split_stratified_into_train_val_test` in the module `toolos` from the `utils` directory.<br>



In [ ]:
train_dataset, valid_dataset, test_dataset = split_stratified_into_train_val_test((images, labels), 
                                                                                  frac_train=0.8, 
                                                                                  frac_val=0.1,
                                                                                  frac_test=0.1, 
                                                                                  shuffle=False,
                                                                                  seed=SEED)
# Extract image and label arrays from the datasets:
train_img, train_lab = train_dataset
valid_img, valid_lab = valid_dataset
test_img, test_lab   = test_dataset

### Data set parameters:

To avoid hard-coding the number of training, valid and test images as well as the size of the images, these parameters are recovered from the data set:
- with the shape attribute of the `train_img` and `test_img` arrays
- with the size attribute of the first training image for example


In [ ]:
NB_TRAIN_IMG = train_img.shape[0] # number of training images
NB_VALID_IMG = valid_img.shape[0] # number of validation images 
NB_TEST_IMG  = test_img.shape[0]  # number of test images

NB_PIXEL     = train_img[0].size  # number of elements (pixels) of the firts training image: 

# Display checking:
print(f"{NB_TRAIN_IMG} training images, {NB_VALID_IMG} validation images and {NB_TEST_IMG} test images")
print(f"{train_img.shape[1]}x{train_img.shape[2]}={NB_PIXEL} pixels in each image")

# number of classes:
NB_CLASS = len(set(train_lab))
print(f"{NB_CLASS} classes found in the `train_lab` ndarray")

#### Check: display the shapes of the dataset arrays

In [ ]:
print(f'train:  {train_img.shape}')
print(f'valid:  {valid_img.shape}')
print(f'test :  {test_img.shape}')

In [ ]:
prop = {}
prop['valid'] = [ (valid_lab == i).sum() for i in range(NB_CLASS)]
prop['test']  = [ (test_lab  == i).sum() for i in range(NB_CLASS)]
plot_proportion_bar(prop, range(NB_CLASS));

#### Check: display a train image randomy selected

In [ ]:
img_rank = np.random.randint(0, len(train_img))

plt.figure(figsize=(4,4))
plt.imshow(train_img[img_rank], cmap='gray')
plt.title(f'Image n° {img_rank}: \n label: {train_lab[img_rank]}, titre: {label_text[train_lab[img_rank]]}',
          fontsize=10)
plt.axis('off');

### Updating the dimensions of the input data for the _keras_ module:

The convolutional layers of the *kera*s module expect 4-dimensional arrays `(batch_size, height, width, depth)` by default:
- `batch_size`: number of input images,
- `height` and `width`: height and width of images (in pixels),
- `depth`: depth of the arrays (`3` for an RGB image, `1` for a grayscale image).

The form of our images is:

In [ ]:
train_img.shape, valid_img.shape, test_img.shape

It is therefore necessary to add a dimension (equal to `1`) after the third dimension `520`, for example with the `reshape` method of the `ndarray` arrays of numpy, without forgetting to divide the arrays by 255 to __normalize__ them entries:

In [ ]:
# avec  la méthode reshape des tableaux ndarray de numpy :
x_train = train_img.reshape(train_img.shape + (1,))/255.
x_valid = valid_img.reshape(valid_img.shape + (1,))/255
x_test  = test_img.reshape(test_img.shape + (1,))/255

Check:

In [ ]:
train_img.shape, x_train.shape, x_train.min(), x_train.max(),

In [ ]:
test_img.shape, x_test.shape, x_test.min(), x_test.max()

### One-hot formatting of labels

The labels data must be given as an array of vectors _one-hot_ coding the integer value of the labels:

In [ ]:
y_train = to_categorical(train_lab)
y_valid = to_categorical(valid_lab)
y_test  = to_categorical(test_lab)

#### Check: display the first 10 values of `train_lab` and `y_train` :

In [ ]:
print(train_lab[:10])
print(y_train[:10])

## 4 - Build the convolutional neural network <a name="4"></a>

### 4.1 - The convolution operation

### Extracting features from an image with a convolution filter

The convolution of an image by a filter (also called kernel, *kernel*) consists of moving a _small 2D window_ (3x3, 5x5....) over the pixels of the image and calculating each time the _contracted tensor product_ between the elements of the filter and the pixels of the image delimited by the filter window (sum of the products term by term).<br>

The animation below illustrates the convolution of a 5x5 image by a 3x3 filter without *padding* on the edges: we obtain a new, smaller image of 3x3 pixels<br>
<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/filter_3x3.png" width="80" style="display:inline-block;">
     <img src="img/Convolution_schematic.gif" width="300" style="display:inline-block;"><br>
     [image credit: <a href="http://deeplearning.stanford.edu/tutorial">Stanford deep learning tutorial</A>]
</p>

To maintain the size of the input image, we can use a *padding* technique to create new data on the edges of the image (by duplicating data on the edges, or adding rows and columns of 0... for example):

<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/padding.gif" width="350"><br>
     [image credit: <a href="https://towardsdatascience.com/applied-deep-learning-part-4-convolutional-neural-networks-584bc134c1e2">Arden Dertat</a> ]
</p>

The goal of convolution is to extract features present in the source image: we speak of a “feature map” to designate the image produced by the convolution operation. The state of the art leads to using several convolutional filters to extract different characteristics: we can have up to several dozen convolutional filters in the same layer of the network which generate as many _feature maps_, hence the increase in data created by these convolution operations...

#### Examples of feature extraction with known convolutional filters ([Prewitt filter](https://fr.wikipedia.org/wiki/Filtre_de_Prewitt)):

As an example, the figure below shows the *feature maps* obtained by convolving a MNIST image (a digit 7) with 4 3x3 filters well known in image processing (Prewitt filters for contour extraction) :

<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/7_mnist_4_filtres.png" width="500"><br>
     [image credit: JLC]
</p>

We see that these filters act as edge detection filters: in the output images, the whitest pixels constitute what the filters detected:
- filters (a) and (c) detect lower and upper horizontal contours,
- filters (b) and (d) detect right and left vertical contours.

These very simple examples allow you to understand how the extraction of *features* from an image using convolutional filtering works.

#### From the convolutional filter to the convolutional neuron layer

The integration of convolutional filtering into the structure of the neural network gives the following organization of calculations:

- Each convolutional filter has the same coefficients for the 3 colors: for the LeNet5 network for example, each of the 6 5x5x3 filters of the first layer has only 25 coefficients, identical for the colors R, G & B.

- Each unit (convolutional neuron) of a *feature map* of layer C1 receives 75 pixels (25 red pixels $R_i$, 25 green pixels $G_i$ and 25 blue pixels $B_i$) delimited by the position of the convolutional filter in the source image.

- The neuron $k$ of a *feature map* calculates an output $y_k = F_a(\sum_{i=1}^{25}{\omega_i(R_i + G_i + B_i) - b_k})$, where $ b_k$ is the bias of the neuron $k$ and $F_a$ the activation function (very often `relu`).

- for the 6 convolutional filters of layer C1, we therefore have 6 x (25 + 1) parameters, i.e. 156 unknown parameters for this layer which will be determined by training the network.

The same pattern is used in all convolutional layers.

### 4.2 - Reducing the amount of information with _pooling_

*pooling* aims to reduce the amount of data to be processed. As for the convolution operation, we move a filter over the elements of the *feature map* array and at each position of the filter, we calculate a number representing all the elements selected in the filter (the maximum value, or the average....). But unlike convolution, we move the filter without overlap.<br>
In the simplified example below, the *max spool* filter transforms the 8x8 matrix into a 4x4 matrix which describes "roughly" the same information but with less data:
<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/max_pool_2x2.png" width="350"><br>
     [image credit: JLC</a> ]
</p>

### 4.3 - Build the "LeNet5 CNN"  with tensorflow/keras

We build a convolutional NN similar to __LeNet5__ introduced in the research paper “Gradient-Based Learning Applied To Document Recognition” in 1998 by Yann LeCun, Leon Bottou, Yoshua Bengio, and Patrick Haffnerfrom.
LeNet5 was originally designed for 32$\times$32 images:

![img/LeNet5.png](img/LeNet5.png)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Conv2D, MaxPool2D, Flatten

def build_DNN(seed=None):

    if seed is not None:
        ##########################
        # Deterministic training #
        ##########################
        # 1/ set the seed of the random generators involved by tensorflow:
        tf.keras.utils.set_random_seed(seed)
        # 2/ make the tf ops determinisctic 
        # [see https://blog.tensorflow.org/2022/05/whats-new-in-tensorflow-29.html]
        tf.config.experimental.enable_op_determinism() 

    model = Sequential(name='LeNet5')
    model.add(Input(shape=x_train[0].shape))
    model.add(Conv2D(6, kernel_size=5, padding='same', activation='relu', name='C1'))
    model.add(MaxPool2D(pool_size=2, name='S2'))
    model.add(Conv2D(16, kernel_size=5, padding='valid', activation='relu', name='C3'))
    model.add(MaxPool2D(pool_size=2, name='S4'))
    model.add(Conv2D(120, kernel_size=5, padding='valid', activation='relu', name='C5'))
    model.add(Flatten())
    model.add(Dense(84, activation='relu', name='F6'))
    model.add(Dense(NB_CLASS, activation='softmax', name='Output'))
    
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model
    

In [ ]:
model = build_DNN()
model.summary()

### Save the initial state of the network (structure & data)

We can save the weights of the initial untrained network (random values) and its structure with the `model.save` method. <br>
This will be useful later to re-create the network to its initial state if we want to compare dirrefent trainings:

In [ ]:
model = build_DNN(seed=1234)
model.save(Path(model_path, 'lenet5_init_seed1234.keras'))

# display the tree beginning at f'./models/{key}':
tree = scan_dir(f"{model_path}")
print(f'\nFiles in <{model_path}>:\n{tree}')

## 5 $-$ Train the network <a name="5"></a>

### Automatically stop training before over-fit
Keras offers tools to automatically stop learning by monitoring for example the growth of `val_accuracy` or the decrease of `val_loss` from one epoch to another (see the _EarlyStopping_ callback).

We can thus define a list of callback functions that we pass as an argument to the `fit` method with the agument named _callbacks_:

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# define the list of 'callback' fucntions:
callbacks_list = [
    EarlyStopping(monitor='val_loss',  # The parameter to monitor
                  patience=2,          # accept that 'val_accuracy' decrease 1 time
                  restore_best_weights=True,
                  verbose=1)
]

# load the network structure & initial weights:
model = tf.keras.models.load_model(Path(model_path, 'lenet5_init_seed1234.keras'))

# Deterministic tensorflow training: 
# see https://blog.tensorflow.org/2022/05/whats-new-in-tensorflow-29.html
tf.keras.utils.set_random_seed(SEED)  # sets seeds for base-python, numpy and tf
tf.config.experimental.enable_op_determinism() 

hist = model.fit(x_train, y_train,
                 validation_data=(x_valid, y_valid),
                 epochs = 25,     # the total number of successive trainings
                 batch_size = 16, # fragmentation of the whole dada set in batches
                 callbacks  = callbacks_list)

### Display the `accuracy` and `loss` curves:

In [ ]:
plot_loss_accuracy(hist);

### Save the structure and the weights of the trained network

In [ ]:
model.save(Path(model_path, 'lenet5_init_seed1234_bs-16_ep-25_es-loss-2.keras'))

# display the tree beginning at f'./models/{key}':
tree = scan_dir(f"{model_path}")
print(f'\nFiles in <{model_path}>:\n{tree}')

## 6 $-$ Evaluate the trained network <a name="6"></a>

### Show the confusion matrix

We explicitly reload the NN structure & trained weights, even if it is not necessary here, to show the Python syntaxe if needed (this cell could be skipped):

In [ ]:
# load the NN structure & trained weights:
model = tf.keras.models.load_model(Path(model_path, 'lenet5_init_seed1234_bs-16_ep-25_es-loss-2.keras'))

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)

# Predicting labels for test images
predict_labels = np.argmax(model.predict(x_test), axis=-1)

# Display classification report
print("Classification Report:\n", classification_report(np.argmax(y_test, axis=-1), predict_labels))

In [ ]:
test_lab[:30]

### Look at the classification score for some test images:

In [ ]:
fig, axes = plt.subplots(2, 3, layout="constrained")
fig.set_size_inches((10,5))

for axis, n in zip(axes.flatten(), (1, 0, 2, 5, 4, 6)):
    results = model.predict(x_test[n:n+1])
    answer  = results.argmax()

    axis.imshow(x_test[n], cmap='gray')
    title = f'\n Image #{n}, label:{test_lab[n]}\n{label_text[test_lab[n]]}'
    title += f'\n\n NN result: {answer} [score:{results[0, answer]*100:.1f}%]'
    axis.set_title(title, fontsize=10)
    axis.axis('off')
    axis.axis('equal')  

In [ ]:
show_conf_matrix(test_lab, predict_labels, range(NB_CLASS), figsize=(7,6));